In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/thuyduc/home-credit-default-risk-sample/bureau_balance_sample.csv
/kaggle/input/datasets/thuyduc/home-credit-default-risk-sample/HomeCredit_columns_description.csv
/kaggle/input/datasets/thuyduc/home-credit-default-risk-sample/instalments_payments_sample.csv
/kaggle/input/datasets/thuyduc/home-credit-default-risk-sample/POS_CASH_balance_sample.csv
/kaggle/input/datasets/thuyduc/home-credit-default-risk-sample/bureau_sample.csv
/kaggle/input/datasets/thuyduc/home-credit-default-risk-sample/previous_application_sample.csv
/kaggle/input/datasets/thuyduc/home-credit-default-risk-sample/application_train_sample.csv
/kaggle/input/datasets/thuyduc/home-credit-default-risk-sample/credit_card_balance_sample.csv
/kaggle/input/datasets/thuyduc/columns-final/columns_final.csv


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import matplotlib.ticker as mticker
import math
import itertools
from typing import Literal
from matplotlib.ticker import FuncFormatter
from matplotlib.axes import Axes
from matplotlib.figure import Figure
import seaborn.objects as so
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [3]:
from sklearn import set_config
set_config(transform_output='pandas', display='diagram')

# Notebook Purpose

Trong notebook này, nghiên cứu cách xây dựng các data pipeline cho các tập dữ liệu. Các Pipeline này có thể tự động loại bỏ các cột không cần thiết, thêm prefix, có thể tự động encoding các biến categorical từ các bảng, và đối với các bảng phụ, pipeline có thể tự tổng hợp (aggregate) theo cấp độ SK_ID_CURR, sau đó một Pipeline khác được thiết kế để merge các tập dữ liệu đã được xử lý thành một final dataset được dùng cho mô hình.

Dataset này lại được thông qua một Feature Engineering Pipeline để tạo ra model-ready dataset trước khi cho vào mô hình ML rồi fine-tune bằng GridSearchCV

Tất cả các Pipeline được thiết kết sao cho có thể fit train, transform test, từ đó đảm bảo tính đúng đắn - test dataset không bị đụng vào

**Cần chỉnh sửa**: Hiện tại thì, đối với việc loại bỏ các feature không cần thiết (feature selection), thì các features này được quyết định dựa trên EDA của toàn bộ dữ liệu nên có thể tiềm ẩn rủi ro Leakage. Do đó, cần xác định các feature không cần thiết chỉ dựa trên tập train data, rồi transform trên test. Các features này nên được detect tự động thay vì làm thủ công

Đây chỉ là Data Pipeline bản thử nghiệm cũng như minh hoạ cách dùng các Pipeline này trong việc train model

In [4]:
application_train = pd.read_csv('/kaggle/input/datasets/thuyduc/home-credit-default-risk-sample/application_train_sample.csv')
bureau = pd.read_csv('/kaggle/input/datasets/thuyduc/home-credit-default-risk-sample/bureau_sample.csv')
bureau_balance = pd.read_csv('/kaggle/input/datasets/thuyduc/home-credit-default-risk-sample/bureau_balance_sample.csv')
credit_card_balance = pd.read_csv('/kaggle/input/datasets/thuyduc/home-credit-default-risk-sample/credit_card_balance_sample.csv')
installments_payments = pd.read_csv('/kaggle/input/datasets/thuyduc/home-credit-default-risk-sample/instalments_payments_sample.csv')
previous_application = pd.read_csv('/kaggle/input/datasets/thuyduc/home-credit-default-risk-sample/previous_application_sample.csv')
POS_CASH_balance = pd.read_csv('/kaggle/input/datasets/thuyduc/home-credit-default-risk-sample/POS_CASH_balance_sample.csv')

In [5]:
from sklearn.model_selection import train_test_split

X = application_train.drop(columns=['TARGET'])
y = application_train['TARGET']

# chia tập dữ liệu cho application_train
X_train, X_val, y_train, y_val = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X_train, y_train, stratify=y_train, test_size=0.2, random_state=42)

application_train_train = pd.concat([X_train, y_train], axis=1)
application_train_val = pd.concat([X_val, y_val], axis=1)
application_train_test = pd.concat([X_test, y_test], axis=1)

SK_ID_CURR_train = X_train['SK_ID_CURR'].values
SK_ID_CURR_val = X_val['SK_ID_CURR'].values
SK_ID_CURR_test = X_test['SK_ID_CURR'].values

# chia tập dữ liệu cho bureau
bureau_train = bureau[bureau['SK_ID_CURR'].isin(SK_ID_CURR_train)]
bureau_val = bureau[bureau['SK_ID_CURR'].isin(SK_ID_CURR_val)]
bureau_test = bureau[bureau['SK_ID_CURR'].isin(SK_ID_CURR_test)]

SK_ID_BUREAU_train = bureau_train['SK_ID_BUREAU'].values
SK_ID_BUREAU_val = bureau_val['SK_ID_BUREAU'].values
SK_ID_BUREAU_test = bureau_test['SK_ID_BUREAU'].values

# chia tập dữ liệu cho previous_application
previous_application_train = previous_application[previous_application['SK_ID_CURR'].isin(SK_ID_CURR_train)]
previous_application_val = previous_application[previous_application['SK_ID_CURR'].isin(SK_ID_CURR_val)]
previous_application_test = previous_application[previous_application['SK_ID_CURR'].isin(SK_ID_CURR_test)]

SK_ID_PREV_train = previous_application_train['SK_ID_PREV'].values
SK_ID_PREV_val = previous_application_val['SK_ID_PREV'].values
SK_ID_PREV_test = previous_application_test['SK_ID_PREV'].values

# chia tập dữ liệu cho bureau_balance
bureau_balance_train = bureau_balance[bureau_balance['SK_ID_BUREAU'].isin(SK_ID_BUREAU_train)]
bureau_balance_val = bureau_balance[bureau_balance['SK_ID_BUREAU'].isin(SK_ID_BUREAU_val)]
bureau_balance_test = bureau_balance[bureau_balance['SK_ID_BUREAU'].isin(SK_ID_BUREAU_test)]

# chia tập dữ liệu cho POS_CASH_balance
filter_pos_cash_train = (POS_CASH_balance['SK_ID_CURR'].isin(SK_ID_CURR_train)) & (POS_CASH_balance['SK_ID_PREV'].isin(SK_ID_PREV_train))
filter_pos_cash_val = (POS_CASH_balance['SK_ID_CURR'].isin(SK_ID_CURR_val)) & (POS_CASH_balance['SK_ID_PREV'].isin(SK_ID_PREV_val))
filter_pos_cash_test = (POS_CASH_balance['SK_ID_CURR'].isin(SK_ID_CURR_test)) & (POS_CASH_balance['SK_ID_PREV'].isin(SK_ID_PREV_test))

POS_CASH_balance_train = POS_CASH_balance[filter_pos_cash_train]
POS_CASH_balance_val = POS_CASH_balance[filter_pos_cash_val]
POS_CASH_balance_test = POS_CASH_balance[filter_pos_cash_test]

# chia tập dữ liệu cho instalments_payments
filter_instalments_train = (installments_payments['SK_ID_CURR'].isin(SK_ID_CURR_train)) & (installments_payments['SK_ID_PREV'].isin(SK_ID_PREV_train))
filter_instalments_val = (installments_payments['SK_ID_CURR'].isin(SK_ID_CURR_val)) & (installments_payments['SK_ID_PREV'].isin(SK_ID_PREV_val))
filter_instalments_test = (installments_payments['SK_ID_CURR'].isin(SK_ID_CURR_test)) & (installments_payments['SK_ID_PREV'].isin(SK_ID_PREV_test))

installments_payments_train = installments_payments[filter_instalments_train]
installments_payments_val = installments_payments[filter_instalments_val]
installments_payments_test = installments_payments[filter_instalments_test]

# chia tập dữ liệu cho credit_card_balance
filter_credit_train = (credit_card_balance['SK_ID_CURR'].isin(SK_ID_CURR_train)) & (credit_card_balance['SK_ID_PREV'].isin(SK_ID_PREV_train))
filter_credit_val = (credit_card_balance['SK_ID_CURR'].isin(SK_ID_CURR_val)) & (credit_card_balance['SK_ID_PREV'].isin(SK_ID_PREV_val))
filter_credit_test = (credit_card_balance['SK_ID_CURR'].isin(SK_ID_CURR_test)) & (credit_card_balance['SK_ID_PREV'].isin(SK_ID_PREV_test))

credit_card_balance_train = credit_card_balance[filter_credit_train]
credit_card_balance_val = credit_card_balance[filter_credit_val]
credit_card_balance_test = credit_card_balance[filter_credit_test]

In [6]:
# một số cột sẽ bị loại bỏ vì tỷ lệ missing values quá cao và một số cột không có nhiều thông tin vì dữ liệu rất lệch - frequency (density) nghiêng rất nhiều về một giá trị cụ thể
# đây đang là detect thủ công và dựa trên EDA của toàn bộ dữ liệu nên cần làm sao để nó tự detect dựa trên một threshold nào đó

columns_remove_application_train = ['NONLIVINGAPARTMENTS_MEDI', 'AMT_REQ_CREDIT_BUREAU_QRT', 'ELEVATORS_MEDI', 'FLOORSMAX_AVG', 'NONLIVINGAPARTMENTS_MODE',\
                                    'APARTMENTS_MEDI','ENTRANCES_MODE','EXT_SOURCE_1','AMT_REQ_CREDIT_BUREAU_DAY','FLOORSMAX_MODE','NONLIVINGAREA_MODE',\
                                    'LIVINGAREA_MEDI','OCCUPATION_TYPE','YEARS_BUILD_MEDI','AMT_REQ_CREDIT_BUREAU_MON','NONLIVINGAPARTMENTS_AVG',\
                                    'APARTMENTS_AVG','FLOORSMIN_MEDI','COMMONAREA_MODE','BASEMENTAREA_MEDI','WALLSMATERIAL_MODE','LANDAREA_MEDI',\
                                    'LIVINGAPARTMENTS_MEDI','LANDAREA_MODE','TOTALAREA_MODE','OWN_CAR_AGE','BASEMENTAREA_MODE','AMT_REQ_CREDIT_BUREAU_HOUR',\
                                    'ENTRANCES_MEDI','HOUSETYPE_MODE','LIVINGAPARTMENTS_MODE','NONLIVINGAREA_MEDI','EXT_SOURCE_3','FLOORSMIN_AVG','LIVINGAPARTMENTS_AVG',\
                                    'FLOORSMIN_MODE','LIVINGAREA_MODE','FONDKAPREMONT_MODE','ELEVATORS_MODE','LIVINGAREA_AVG','COMMONAREA_MEDI','YEARS_BEGINEXPLUATATION_AVG',\
                                    'BASEMENTAREA_AVG','YEARS_BEGINEXPLUATATION_MODE','AMT_REQ_CREDIT_BUREAU_WEEK','YEARS_BEGINEXPLUATATION_MEDI','FLOORSMAX_MEDI','ELEVATORS_AVG',\
                                    'APARTMENTS_MODE','YEARS_BUILD_AVG','AMT_REQ_CREDIT_BUREAU_YEAR','YEARS_BUILD_MODE','NONLIVINGAREA_AVG','ENTRANCES_AVG','LANDAREA_AVG','EMERGENCYSTATE_MODE',\
                                    'COMMONAREA_AVG','FLAG_MOBIL','FLAG_CONT_MOBILE','FLAG_EMAIL','REG_REGION_NOT_LIVE_REGION','REG_REGION_NOT_WORK_REGION','LIVE_REGION_NOT_WORK_REGION',\
                                    'REG_CITY_NOT_LIVE_CITY','FLAG_DOCUMENT_2','FLAG_DOCUMENT_4','FLAG_DOCUMENT_5','FLAG_DOCUMENT_6','FLAG_DOCUMENT_7','FLAG_DOCUMENT_8','FLAG_DOCUMENT_9',\
                                    'FLAG_DOCUMENT_10','FLAG_DOCUMENT_11','FLAG_DOCUMENT_12','FLAG_DOCUMENT_13','FLAG_DOCUMENT_14','FLAG_DOCUMENT_15','FLAG_DOCUMENT_16','FLAG_DOCUMENT_17',\
                                    'FLAG_DOCUMENT_18','FLAG_DOCUMENT_19','FLAG_DOCUMENT_20','FLAG_DOCUMENT_21','WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE']


columns_remove_bureau = ['AMT_CREDIT_SUM_DEBT','AMT_CREDIT_MAX_OVERDUE','DAYS_ENDDATE_FACT','AMT_CREDIT_SUM_LIMIT','AMT_ANNUITY',\
                         'CREDIT_CURRENCY','CREDIT_DAY_OVERDUE','CNT_CREDIT_PROLONG']



columns_remove_credit_card = ['AMT_DRAWINGS_POS_CURRENT','CNT_DRAWINGS_POS_CURRENT','AMT_DRAWINGS_OTHER_CURRENT','CNT_DRAWINGS_ATM_CURRENT','AMT_DRAWINGS_ATM_CURRENT',\
                              'AMT_PAYMENT_CURRENT','CNT_DRAWINGS_OTHER_CURRENT','SK_DPD','SK_DPD_DEF']



columns_remove_previous_application = ['DAYS_FIRST_DUE','NFLAG_INSURED_ON_APPROVAL','RATE_DOWN_PAYMENT','DAYS_TERMINATION','NAME_TYPE_SUITE','DAYS_LAST_DUE','RATE_INTEREST_PRIVILEGED',\
                                       'AMT_GOODS_PRICE','DAYS_FIRST_DRAWING','CNT_PAYMENT','DAYS_LAST_DUE_1ST_VERSION','AMT_DOWN_PAYMENT','AMT_ANNUITY','RATE_INTEREST_PRIMARY',\
                                       'NFLAG_LAST_APPL_IN_DAY','FLAG_LAST_APPL_PER_CONTRACT','WEEKDAY_APPR_PROCESS_START','HOUR_APPR_PROCESS_START']


columns_remove_pos_cash_balance = ['SK_DPD', 'SK_DPD_DEF']



# Định nghĩa các Custom Transformer

In [7]:
class RemoveFeature(BaseEstimator, TransformerMixin):
    def __init__(self, columns_removed:list[str]):
        """
        columns_removed: columns which will be removed from dataset
        """
        self.columns_removed = columns_removed

    def fit(self, X:pd.DataFrame, y=None):
        missing = set(self.columns_removed) - set(X.columns) # check if columns's name removed is misspelled
        if missing:
            raise ValueError(f"Các cột sau không tồn tại trong dữ liệu: {missing}")
        
        # Giữ nguyên thứ tự cột gốc, chỉ loại bỏ cột cần xóa
        self.columns_selected_ = [
            col for col in X.columns if col not in self.columns_removed
        ]
        return self

    def transform(self, X:pd.DataFrame):
        X_copy = X.copy()
        X_new = X_copy[self.columns_selected_]
        return X_new

    def get_feature_names_out(self, input_features=None): # must have this method
        # must have 'input_features' parameter
        return self.columns_selected_

In [8]:
class AddPrefix(BaseEstimator, TransformerMixin):
    def __init__(self, prefix:str, exclude:list[str]):
        """
        Used to add prefix to columns' name
        
        Args:
            prefix: prefix added to columns' name
            exclude: columns that don't need prefix
        """
        self.prefix = prefix
        self.exclude = exclude

    def fit(self, X:pd.DataFrame, y=None):
        self.new_cols_name_ = [
            col if col in self.exclude else f'{self.prefix}_{col}'
            for col in X.columns
        ]
        return self

    def transform(self, X:pd.DataFrame):
        X_copy = X.copy()
        X_copy.columns = self.new_cols_name_
        return X_copy

    def get_feature_names_out(self, input_features=None): # must have this method
        # must have 'input_features' parameter
        return self.new_cols_name_

In [9]:
class Aggeration(BaseEstimator, TransformerMixin):
    def __init__(self, cat_cols:list[str], num_cols:list[str], groupby_col:str):
        """
        Used to perform aggregation by specific columns, num_cols and cat_cols will have differenct aggregation
        
        Args:
            cat_cols: categorical column's name
            num_cols: numerical column's name
            groupby_col: specific column's name used for groupby to perform aggregation
        """
        if groupby_col in num_cols:
            num_cols.remove(groupby_col)

        self.cat_cols = cat_cols
        self.num_cols = num_cols
        self.groupby_col = groupby_col

    def _build_agg_dict(self):
        agg_dict = {}
        for col in self.cat_cols:
            agg_dict[col] = ['sum', 'mean']
        for col in self.num_cols:
            agg_dict[col] = ['min', 'max', 'mean', 'sum'] # ['min', 'max', 'mean']
        return agg_dict
    
    def fit(self, X:pd.DataFrame, y=None):
        agg_dict = self._build_agg_dict()
        self.new_cols_name_ = [self.groupby_col] + [
            f'{col}_{stat.upper()}' for col, stats in agg_dict.items() for stat in stats
        ]
        return self

    def transform(self, X:pd.DataFrame):
        X_copy = X.copy()
        agg_dict = self._build_agg_dict()

        X_agg = X_copy.groupby(self.groupby_col).agg(agg_dict)
        X_agg.columns = self.new_cols_name_[1:]
        X_agg.reset_index(inplace=True)
        
        return X_agg

    def get_feature_names_out(self, input_features=None): # must have this method
        # must have 'input_features' parameter
        return self.new_cols_name_

# Xây dựng Pipeline xử lý cho từng tập dữ liệu

In [10]:
class BureauBalancePipeline(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X:pd.DataFrame, y=None):
        X_copy = X.copy()
        bureau_balance_add_prefix = AddPrefix(prefix='BB', exclude=['SK_ID_BUREAU'])
        self.pipe_ = Pipeline([
            ('rename_columns', bureau_balance_add_prefix)
        ])
        X_copy = self.pipe_.fit_transform(X_copy)
        
        cat_cols = list(X_copy.select_dtypes(include=['object']).columns)
        num_cols = list(X_copy.select_dtypes(include=['number']).columns)
        
        self.cat_cols_transformer_ = ColumnTransformer(
            transformers=[
                ('one_hot_encode', OneHotEncoder(sparse_output=False, drop='if_binary', handle_unknown='ignore'), cat_cols)
            ],
            remainder='passthrough',
            verbose_feature_names_out=False, # not containing the transformer name in columns of Dataframe
            # force_int_remainder_cols=False
        )
        X_copy = self.cat_cols_transformer_.fit_transform(X_copy)

        cat_cols_after_onehot = [col for col in X_copy.columns if col.startswith(tuple([f"{column}_" for column in cat_cols]))]
        self.aggregate_ = Aggeration(cat_cols=cat_cols_after_onehot, num_cols=num_cols, groupby_col='SK_ID_BUREAU')
        self.aggregate_.fit(X_copy)
        
        return self

    def transform(self, X:pd.DataFrame):
        X_copy = X.copy()
        X_final = self.pipe_.transform(X_copy)
        X_final = self.cat_cols_transformer_.transform(X_final)
        X_final = self.aggregate_.transform(X_final)
        
        return X_final

    def get_feature_names_out(self, input_features=None):
        return self.aggregate_.get_feature_names_out()

In [11]:
class BureauPipeline(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X:pd.DataFrame, y=None):
        X_copy = X.copy()
        bureau_add_prefix = AddPrefix(prefix='B', exclude=['SK_ID_CURR', 'SK_ID_BUREAU'])
        feature_removed_bureau = RemoveFeature(columns_removed = columns_remove_bureau)

        self.pipe_ = Pipeline([
            ('remove_features', feature_removed_bureau),
            ('rename_columns', bureau_add_prefix)
        ])
        X_copy = self.pipe_.fit(X_copy)
        
        return self

    def transform(self, X:pd.DataFrame):
        X_copy = X.copy()
        X_final = self.pipe_.transform(X_copy)
        return X_final

    def get_feature_names_out(self, input_features=None):
        return self.pipe_.get_feature_names_out()

In [12]:
class InstallmentPaymentPipeline(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X:pd.DataFrame, y=None):
        X_copy = X.copy()
        X_copy.drop(columns=['SK_ID_CURR'], inplace=True)
        
        installment_payment_add_prefix = AddPrefix(prefix='IP', exclude=['SK_ID_PREV'])
        self.pipe_ = Pipeline([
            ('rename_columns', installment_payment_add_prefix)
        ])
        X_copy = self.pipe_.fit_transform(X_copy)

        cat_cols = list(X_copy.select_dtypes(include=['object']).columns) # empty list
        num_cols = list(X_copy.select_dtypes(include=['number']).columns)

        self.aggregate_ = Aggeration(cat_cols=[], num_cols=num_cols, groupby_col='SK_ID_PREV')
        self.aggregate_.fit(X_copy)

        return self

    def transform(self, X:pd.DataFrame):
        X_copy = X.copy()
        X_copy.drop(columns=['SK_ID_CURR'], inplace=True)
        X_final = self.pipe_.transform(X_copy)
        X_final = self.aggregate_.transform(X_final)
        return X_final

    def get_feature_names_out(self, input_features=None):
        return self.aggregate_.get_feature_names_out()

In [13]:
class POSCASHPipeline(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X:pd.DataFrame, y=None):
        X_copy = X.copy()
        X_copy.drop(columns=['SK_ID_CURR'], inplace=True)
        
        pos_cash_add_prefix = AddPrefix(prefix='PC', exclude=['SK_ID_PREV'])
        feature_removed_pos_cash_balance = RemoveFeature(columns_removed = columns_remove_pos_cash_balance)

        self.pipe_ = Pipeline([
            ('remove_features',feature_removed_pos_cash_balance),
            ('rename_columns', pos_cash_add_prefix)
        ])
        
        X_copy = self.pipe_.fit_transform(X_copy)

        cat_cols = list(X_copy.select_dtypes(include=['object']).columns)
        num_cols = list(X_copy.select_dtypes(include=['number']).columns)
        
        self.cat_cols_transformer_ = ColumnTransformer(
            transformers=[
                ('one_hot_encode', OneHotEncoder(sparse_output=False, drop='if_binary', handle_unknown='ignore'), cat_cols)
            ],
            remainder='passthrough',
            verbose_feature_names_out=False, # not containing the transformer name in columns of Dataframe
            # force_int_remainder_cols=False
        )
        X_copy = self.cat_cols_transformer_.fit_transform(X_copy)

        cat_cols_after_onehot = [col for col in X_copy.columns if col.startswith(tuple([f"{column}_" for column in cat_cols]))]
        self.aggregate_ = Aggeration(cat_cols=cat_cols_after_onehot, num_cols=num_cols, groupby_col='SK_ID_PREV')
        self.aggregate_.fit(X_copy)
        
        return self

    def transform(self, X:pd.DataFrame):
        X_copy = X.copy()
        X_copy.drop(columns=['SK_ID_CURR'], inplace=True)
        
        X_final = self.pipe_.transform(X_copy)
        X_final = self.cat_cols_transformer_.transform(X_final)
        X_final = self.aggregate_.transform(X_final)
        
        return X_final

    def get_feature_names_out(self, input_features=None):
        return self.aggregate_.get_feature_names_out()

In [14]:
class CreditCardPipeline(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X:pd.DataFrame, y=None):
        X_copy = X.copy()
        X_copy.drop(columns=['SK_ID_CURR'], inplace=True)
        
        credit_card_add_prefix = AddPrefix(prefix='CC', exclude=['SK_ID_PREV'])
        feature_removed_credit_card = RemoveFeature(columns_removed = columns_remove_credit_card)

        self.pipe_ = Pipeline([
            ('remove_features',feature_removed_credit_card),
            ('rename_columns', credit_card_add_prefix)
        ])
        
        X_copy = self.pipe_.fit_transform(X_copy)

        cat_cols = list(X_copy.select_dtypes(include=['object']).columns)
        num_cols = list(X_copy.select_dtypes(include=['number']).columns)
        
        self.cat_cols_transformer_ = ColumnTransformer(
            transformers=[
                ('one_hot_encode', OneHotEncoder(sparse_output=False, drop='if_binary', handle_unknown='ignore'), cat_cols)
            ],
            remainder='passthrough',
            verbose_feature_names_out=False, # not containing the transformer name in columns of Dataframe
            # force_int_remainder_cols=False
        )
        X_copy = self.cat_cols_transformer_.fit_transform(X_copy)

        cat_cols_after_onehot = [col for col in X_copy.columns if col.startswith(tuple([f"{column}_" for column in cat_cols]))]
        self.aggregate_ = Aggeration(cat_cols=cat_cols_after_onehot, num_cols=num_cols, groupby_col='SK_ID_PREV')
        self.aggregate_.fit(X)
        
        return self

    def transform(self, X:pd.DataFrame):
        X_copy = X.copy()
        X_copy.drop(columns=['SK_ID_CURR'], inplace=True)
        
        X_final = self.pipe_.transform(X_copy)
        X_final = self.cat_cols_transformer_.transform(X_final)
        X_final = self.aggregate_.transform(X_final)
        
        return X_final

    def get_feature_names_out(self, input_features=None):
        return self.aggregate_.get_feature_names_out()

In [15]:
class PreviousApplicationPipeline(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X:pd.DataFrame, y=None):
        X_copy = X.copy()
        previous_application_add_prefix = AddPrefix(prefix='PA', exclude=['SK_ID_CURR', 'SK_ID_PREV'])
        feature_removed_previous_application = RemoveFeature(columns_removed = columns_remove_previous_application)

        self.pipe_ = Pipeline([
            ('remove_features', feature_removed_previous_application),
            ('rename_columns', previous_application_add_prefix)
        ])
        self.pipe_.fit(X_copy)

        return self

    def transform(self, X:pd.DataFrame):
        X_copy = X.copy()
        X_final = self.pipe_.transform(X_copy)
        return X_final

    def get_feature_names_out(self, input_features=None):
        return self.pipe_.get_feature_names_out()

In [16]:
class ApplicationTrainPipeline(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X:pd.DataFrame, y=None):
        X_copy = X.copy()
        application_train_add_prefix = AddPrefix(prefix='AT', exclude=['SK_ID_CURR'])
        feature_removed_application_train = RemoveFeature(columns_removed = columns_remove_application_train)

        self.pipe_ = Pipeline([
            ('remove_features', feature_removed_application_train),
            ('rename_columns', application_train_add_prefix)
        ])
        self.pipe_.fit(X_copy)

        return self

    def transform(self, X:pd.DataFrame):
        X_copy = X.copy()
        X_final = self.pipe_.transform(X_copy)
        return X_final

    def get_feature_names_out(self, input_features=None):
        return self.pipe_.get_feature_names_out()

# Xây MergePipeline

MergePipeline có tác dụng merge các tập dữ liệu lại theo SK_ID_CURR, SK_ID_PREV, SK_ID_BUREAU để tạo ra một final dataset mà mỗi dòng là một đơn vay của một khách hàng riêng biệt

In [17]:
class MergePipeline:
    def __init__(self):
        self.bureau_balance_pipeline = BureauBalancePipeline()
        self.installment_payment_pipeline = InstallmentPaymentPipeline()
        self.pos_cash_pipeline = POSCASHPipeline()
        self.credit_card_pipeline = CreditCardPipeline()
        self.previous_application_pipeline = PreviousApplicationPipeline()
        self.bureau_pipeline = BureauPipeline()
        self.application_pipeline = ApplicationTrainPipeline()

        # Lưu các transformer/aggregator theo tên nhóm, thay vì tạo nhiều thuộc tính rời rạc
        self.group_transformers_ = {}

    def _encode_and_aggregate_fit(self, df: pd.DataFrame, group_name: str, groupby_col: str = 'SK_ID_CURR'):
        """
        Dùng khi fit: one-hot encode + aggregate cho 1 nhóm bảng (bureau hoặc previous_application),
        đồng thời lưu lại các transformer đã fit để dùng lại lúc transform.
        """
        cat_cols = list(df.select_dtypes(include=['object']).columns)
        num_cols = list(df.select_dtypes(include=['number']).columns)

        cat_transformer = ColumnTransformer(
            transformers=[
                ('one_hot_encode', OneHotEncoder(sparse_output=False, drop='if_binary', handle_unknown='ignore'), cat_cols)
            ],
            remainder='passthrough',
            verbose_feature_names_out=False,
        )
        df_encoded = cat_transformer.fit_transform(df)

        cat_cols_after_onehot = [
            col for col in df_encoded.columns
            if col.startswith(tuple(f"{c}_" for c in cat_cols))
        ]

        aggregator = Aggeration(cat_cols=cat_cols_after_onehot, num_cols=num_cols, groupby_col=groupby_col)
        df_agg = aggregator.fit_transform(df_encoded)

        # Lưu lại để transform() dùng sau này
        self.group_transformers_[group_name] = {
            'cat_transformer': cat_transformer,
            'aggregator': aggregator,
        }

        return df_agg

    def _encode_and_aggregate_transform(self, df: pd.DataFrame, group_name: str):
        """Dùng khi transform: áp dụng lại transformer đã fit cho nhóm tương ứng."""
        transformers = self.group_transformers_[group_name]
        df_encoded = transformers['cat_transformer'].transform(df)
        df_agg = transformers['aggregator'].transform(df_encoded)
        return df_agg

    def fit(self, bureau_balance:pd.DataFrame, POS_CASH_balance:pd.DataFrame, application_train:pd.DataFrame,
             credit_card_balance:pd.DataFrame, previous_application:pd.DataFrame, bureau:pd.DataFrame, installments_payments:pd.DataFrame):

        bureau_balance_transform = self.bureau_balance_pipeline.fit_transform(bureau_balance)
        installments_payments_transform = self.installment_payment_pipeline.fit_transform(installments_payments)
        POS_CASH_balance_transform = self.pos_cash_pipeline.fit_transform(POS_CASH_balance)
        credit_card_balance_transform = self.credit_card_pipeline.fit_transform(credit_card_balance)
        previous_application_transform = self.previous_application_pipeline.fit_transform(previous_application)
        bureau_transform = self.bureau_pipeline.fit_transform(bureau)
        application_train_transform = self.application_pipeline.fit_transform(application_train)

        # --- Nhóm bureau ---
        merge_bureau = bureau_transform.merge(bureau_balance_transform, on='SK_ID_BUREAU', how='left')
        merge_bureau.drop(columns=['SK_ID_BUREAU'], inplace=True)
        merge_bureau = self._encode_and_aggregate_fit(merge_bureau, group_name='bureau')

        # --- Nhóm previous_application ---
        merge_prev = previous_application_transform.merge(credit_card_balance_transform, on='SK_ID_PREV', how='left')
        merge_prev = merge_prev.merge(POS_CASH_balance_transform, on='SK_ID_PREV', how='left')
        merge_prev = merge_prev.merge(installments_payments_transform, on='SK_ID_PREV', how='left')
        merge_prev.drop(columns=['SK_ID_PREV'], inplace=True)
        merge_prev = self._encode_and_aggregate_fit(merge_prev, group_name='previous_application')

        # --- Merge cuối cùng ---
        final_dataset = application_train_transform.merge(merge_prev, on='SK_ID_CURR', how='left')
        final_dataset = final_dataset.merge(merge_bureau, on='SK_ID_CURR', how='left')
        # self.final_dataset_ = final_dataset

        return self

    def transform(self, bureau_balance:pd.DataFrame, POS_CASH_balance:pd.DataFrame, application_train:pd.DataFrame,
                   credit_card_balance:pd.DataFrame, previous_application:pd.DataFrame, bureau:pd.DataFrame, installments_payments:pd.DataFrame):

        bureau_balance_transform = self.bureau_balance_pipeline.transform(bureau_balance)
        installments_payments_transform = self.installment_payment_pipeline.transform(installments_payments)
        POS_CASH_balance_transform = self.pos_cash_pipeline.transform(POS_CASH_balance)
        credit_card_balance_transform = self.credit_card_pipeline.transform(credit_card_balance)
        previous_application_transform = self.previous_application_pipeline.transform(previous_application)
        bureau_transform = self.bureau_pipeline.transform(bureau)
        application_train_transform = self.application_pipeline.transform(application_train)

        merge_bureau = bureau_transform.merge(bureau_balance_transform, on='SK_ID_BUREAU', how='left')
        merge_bureau.drop(columns=['SK_ID_BUREAU'], inplace=True)
        merge_bureau = self._encode_and_aggregate_transform(merge_bureau, group_name='bureau')

        merge_prev = previous_application_transform.merge(credit_card_balance_transform, on='SK_ID_PREV', how='left')
        merge_prev = merge_prev.merge(POS_CASH_balance_transform, on='SK_ID_PREV', how='left')
        merge_prev = merge_prev.merge(installments_payments_transform, on='SK_ID_PREV', how='left')
        merge_prev.drop(columns=['SK_ID_PREV'], inplace=True)
        merge_prev = self._encode_and_aggregate_transform(merge_prev, group_name='previous_application')

        final_dataset = application_train_transform.merge(merge_prev, on='SK_ID_CURR', how='left')
        final_dataset = final_dataset.merge(merge_bureau, on='SK_ID_CURR', how='left')

        return final_dataset

In [18]:
import joblib
merge_pipeline = MergePipeline()
merge_pipeline.fit(bureau_balance_train, POS_CASH_balance_train, application_train_train, credit_card_balance_train, \
                   previous_application_train, bureau_train, installments_payments_train)

joblib.dump(merge_pipeline, 'merge_pipeline.joblib')

/tmp/ipykernel_58/2646216863.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_agg.reset_index(inplace=True)


['merge_pipeline.joblib']

In [19]:
final_dataset_train = merge_pipeline.transform(bureau_balance_train, POS_CASH_balance_train, application_train_train, credit_card_balance_train, \
                                               previous_application_train, bureau_train, installments_payments_train)
final_dataset_val = merge_pipeline.transform(bureau_balance_val, POS_CASH_balance_val, application_train_val, credit_card_balance_val, \
                                             previous_application_val, bureau_val, installments_payments_val)
final_dataset_test = merge_pipeline.transform(bureau_balance_test, POS_CASH_balance_test, application_train_test, credit_card_balance_test, \
                                              previous_application_test, bureau_test, installments_payments_test)

/tmp/ipykernel_58/2646216863.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_agg.reset_index(inplace=True)
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/tmp/ipykernel_58/2646216863.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_agg.reset_index(inplace=True)
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: User

In [20]:
class FeatureEngineeringPipeline(BaseEstimator, TransformerMixin):
    def __init__(self, missing_threshold: float = 0.1, imbalance_threshold: float = 0.02):
        self.missing_threshold = missing_threshold
        self.imbalance_threshold = imbalance_threshold

    def fit(self, X: pd.DataFrame, y=None):
        X_copy = X.copy()

        # Bước 1: loại cột có tỷ lệ missing quá cao
        missing_ratio = X_copy.isnull().mean()
        columns_to_remove = list(missing_ratio[missing_ratio > self.missing_threshold].index)
        self.feature_remove_ = RemoveFeature(columns_removed=columns_to_remove)
        X_copy = self.feature_remove_.fit_transform(X_copy)

        # Bước 2: one-hot encode cột categorical (binary riêng, non-binary riêng)
        object_columns = X_copy.select_dtypes('object')
        nunique = object_columns.nunique()
        binary_object_columns = list(nunique[nunique == 2].index)
        nonbinary_object_columns = list(nunique[nunique != 2].index)

        self.final_dataset_transformer_ = ColumnTransformer(
            transformers=[
                ("binary_encoding", OneHotEncoder(sparse_output=False, drop='if_binary', handle_unknown='ignore'), binary_object_columns),
                ("one_hot_encoding", OneHotEncoder(sparse_output=False, drop=None, handle_unknown='ignore'), nonbinary_object_columns)
            ],
            remainder='passthrough',
            verbose_feature_names_out=False,
        ).set_output(transform='pandas')  # đảm bảo luôn là DataFrame, không phụ thuộc global config

        X_copy = self.final_dataset_transformer_.fit_transform(X_copy)

        # Bước 3: loại cột binary bị mất cân bằng nặng (theo cả 2 chiều: gần 0 hoặc gần 1)
        number_columns = X_copy.select_dtypes('number')
        nunique_num = number_columns.nunique()
        binary_columns = list(nunique_num[nunique_num == 2].index)

        binary_means = X_copy[binary_columns].mean()
        binary_columns_remove = list(
            binary_means[(binary_means < self.imbalance_threshold) |
                         (binary_means > 1 - self.imbalance_threshold)].index
        )

        self.binary_feature_remove_ = RemoveFeature(columns_removed=binary_columns_remove)
        X_copy = self.binary_feature_remove_.fit_transform(X_copy)

        return self

    def transform(self, X: pd.DataFrame):
        X_copy = X.copy()
        X_final = self.feature_remove_.transform(X_copy)
        X_final = self.final_dataset_transformer_.transform(X_final)
        X_final = self.binary_feature_remove_.transform(X_final)
        return X_final

    def get_feature_names_out(self, input_features=None):
        return self.binary_feature_remove_.get_feature_names_out()

In [21]:
X_train = final_dataset_train.drop(columns=['AT_TARGET', 'SK_ID_CURR'])
y_train = final_dataset_train['AT_TARGET']

X_val = final_dataset_val.drop(columns=['AT_TARGET', 'SK_ID_CURR'])
y_val = final_dataset_val['AT_TARGET']

X_test = final_dataset_test.drop(columns=['AT_TARGET', 'SK_ID_CURR'])
y_test = final_dataset_test['AT_TARGET']

# Xây dựng mô hình ML để train

In [22]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.model_selection import PredefinedSplit, TimeSeriesSplit
from typing import Literal
from sklearn.model_selection import KFold, StratifiedKFold, LeaveOneOut

In [23]:
class GridSearch:
    def __init__(self, X:pd.DataFrame, y:pd.Series):
        self.x = X
        self.y = y

    def get_results(self, cv_results:dict, best_index:np.int64, scoring:dict) -> pd.DataFrame:

        # Get all the results from the CV and put them in a df
        cv_results = pd.DataFrame(cv_results)
    
        # Isolate the row of the df with the max(mean f1 score)
        best_estimator_results = cv_results.iloc[best_index]
    
        # Extract accuracy, precision, recall, and f1 score from that row        
        table = {score: [] for score in scoring}
        table['type'] = ['train', 'test']
        
        for score in scoring:
            for prefix in ['train', 'test']:
                table[score].append(best_estimator_results[f'mean_{prefix}_{score}'])
                
        # Create table of results
        table = pd.DataFrame(table)
        index_of_test = table[table['type']=='test'].index
        table.loc[index_of_test, 'type'] = 'validate'
        
        return table
    
    def grid_search_cv(self, estimator, params_grid:dict, scoring:dict, 
                        cv: Literal['KFold', 'StratifiedKFold', 'Predefined', 'TimeSeries'],
                       refit:str, fit_params:dict|None=None) -> tuple[dict, pd.DataFrame]:
        
        if cv == 'StratifiedKFold':
            cv = StratifiedKFold(n_splits=3, random_state=42, shuffle=True)
        
        elif cv == 'KFold':
            cv = KFold(n_splits=3, random_state=42, shuffle=True)
        
        elif cv == 'LeaveOneOut':
            cv = LeaveOneOut()
        
        elif cv == 'TimeSeries':
            cv = TimeSeriesSplit(n_splits=5)
        
        elif cv == 'Predefined':
            X_train, X_val, y_train, y_val = train_test_split(self.x, self.y, test_size=0.2, 
                                                          stratify=self.y, random_state=42)
            X_val: pd.DataFrame

            split_index = [0 if x in X_val.index else -1 for x in self.x.index]
            cv = PredefinedSplit(split_index)
        
        search = GridSearchCV(estimator=estimator, param_grid=params_grid, scoring=scoring, 
                              cv=cv, refit=refit, n_jobs=-1, return_train_score=True)
        
        if fit_params is None:
            fit_params = {}
        
        search.fit(self.x, self.y, **fit_params)
        best_index = search.best_index_
        cv_results = search.cv_results_
        
        table = self.get_results(cv_results, best_index, scoring)
        
        best_model = {'best_estimator': search.best_estimator_,
                    'best_params': search.best_params_,}
        
        return best_model, table
    
    def random_search_cv(self, estimator, param_distributions:dict, scoring:dict,
                         cv:Literal['KFold', 'StratifiedKFold', 'Predefined', 'TimeSeries'], 
                         refit:str, fit_params:dict|None=None) -> tuple[dict, pd.DataFrame]:
        
        if cv == 'StratifiedKFold':
            cv = StratifiedKFold(n_splits=3, random_state=42, shuffle=True)
        
        elif cv == 'KFold':
            cv = KFold(n_splits=3, random_state=42, shuffle=True)

        elif cv == 'LeaveOneOut':
            cv = LeaveOneOut()
        
        elif cv == 'TimeSeries':
            cv = TimeSeriesSplit(n_splits=5)
        
        elif cv == 'Predefined':
            X_train, X_val, y_train, y_val = train_test_split(self.x, self.y, test_size=0.2, 
                                                          stratify=self.y, random_state=42)
            
            split_index = [0 if x in X_val.index else -1 for x in self.x.index]
            cv = PredefinedSplit(split_index)

        search = RandomizedSearchCV(estimator=estimator, param_distributions=param_distributions, 
                                    n_iter=10, n_jobs=-1, return_train_score=True,
                                    scoring=scoring, cv=cv, refit=refit)

        if fit_params is None:
            fit_params = {}
        
        search.fit(self.x, self.y, **fit_params)
        best_index = search.best_index_
        cv_results = search.cv_results_
        
        table = self.get_results(cv_results, best_index, scoring)
        
        best_model = {'best_estimator': search.best_estimator_,
                     'best_params': search.best_params_,}
        
        return best_model, table

In [24]:
random_forest = RandomForestClassifier(random_state = 42)
xg_boost = XGBClassifier(objective="binary:logistic", random_state=42)
ada_boost = AdaBoostClassifier(random_state=42)

feature_engineering_pipeline = FeatureEngineeringPipeline()

In [25]:
xg_pipeline = Pipeline([
    ('feature_engineering', feature_engineering_pipeline),
    ('xg', xg_boost)
])

rf_pipeline = Pipeline([
    ('feature_engineering', feature_engineering_pipeline),
    ('rf', random_forest)
])

ada_pipeline = Pipeline([
    ('feature_engineering', feature_engineering_pipeline),
    ('ada', ada_boost)
])

# Fine-tune model với GridSearchCV

In [26]:
X_train_copy = X_train.copy()
X_test_copy = X_test.copy()
X_val_copy = X_val.copy()

In [27]:
searcher = GridSearch(X = X_train_copy, y = y_train)

In [ ]:
y_train

In [31]:
# fine tune cho xg boost
param_distributions = {
    "xg__max_depth": [3, 4, 5],
    "xg__learning_rate": [0.05, 0.1, 1],
    "xg__n_estimators": [300],
    "xg__subsample": [0.2, 0.4, 0.6],
    "xg__colsample_bytree": [0.2],
    "xg__scale_pos_weight": [6.0, 8.0, 10.0]
}

scoring = {'f1': 'f1', 'recall': 'recall', 'precision': 'precision', 'neg_log_loss': 'neg_log_loss'}

xg_best_model, xg_table = searcher.random_search_cv(estimator=xg_pipeline, param_distributions=param_distributions, 
                                               scoring=scoring, cv='StratifiedKFold', refit='neg_log_loss')

xg_best_model

{'best_estimator': Pipeline(steps=[('feature_engineering', FeatureEngineeringPipeline()),
                 ('xg',
                  XGBClassifier(base_score=None, booster=None, callbacks=None,
                                colsample_bylevel=None, colsample_bynode=None,
                                colsample_bytree=0.2, device=None,
                                early_stopping_rounds=None,
                                enable_categorical=False, eval_metric=None,
                                feature_types=None, feature_weights=None,
                                gamma=None, grow_policy=None,
                                importance_type=None,
                                interaction_constraints=None, learning_rate=0.1,
                                max_bin=None, max_cat_threshold=None,
                                max_cat_to_onehot=None, max_delta_step=None,
                                max_depth=5, max_leaves=None,
                                min_child_wei

In [32]:
xg_table

,f1,recall,precision,neg_log_loss,type
0,0.765296,0.871616,0.682093,-0.209890,train
1,0.231620,0.225302,0.238436,-0.330009,validate


In [52]:
import joblib
xg_tuned_model = xg_best_model['best_estimator']
best_path = os.path.join('/kaggle/working', 'xg_tuned_pipeline.joblib')
joblib.dump(xg_tuned_model, best_path)

['/kaggle/working/xg_tuned_pipeline.joblib']

In [38]:
from sklearn.metrics import roc_auc_score

# Tiếp theo ta sẽ đến với đường cong ROC (Receiver operating characteristic)
class ROCCurve:
    def __init__(self, X:pd.DataFrame, y:pd.Series):
        self.x = X
        self.y = y

    def plot_roc_curve_with_thresholds(self, estimators_dict:dict, plots_each_row:int, tree_models:bool):
        num_plots = len(estimators_dict)
        number_rows = math.ceil(num_plots / plots_each_row)
        
        fig, axes = plt.subplots(nrows=number_rows, ncols=plots_each_row, figsize=(16, 8),) # sharey=True
        axes = np.array(axes).reshape(number_rows, plots_each_row)
        coordinates = list(itertools.product(range(number_rows), range(plots_each_row)))

        for coor, (estimator_name, estimator) in zip(coordinates, estimators_dict.items()):
            y_score = estimator.predict_proba(self.x)[:, 1] # certainty measure for the positive class (class 1)
            fpr, tpr, thresholds = roc_curve(self.y, y_score)

            axes[coor].axvline(x=0.5, color="red", linestyle=":", linewidth=2, label=f"{estimator_name}'s Default Threshold")
            axes[coor].plot(thresholds, fpr, "b--", label="FPR")
            axes[coor].plot(thresholds, tpr, "g-", label="TPR (recall)")
            axes[coor].set_xlabel("Threshold")
            axes[coor].set_title(estimator_name)
            axes[coor].legend(loc="best")
            axes[coor].set_ylim([0, 1])
            axes[coor].grid()

        # hide any unused axes
        for coor in coordinates[num_plots:]:
            fig.delaxes(axes[coor])
    
    def plot_roc_curve(self, estimators_dict:dict, plots_each_row:int, tree_models:bool):
        num_plots = len(estimators_dict)
        number_rows = math.ceil(num_plots / plots_each_row)
        
        fig, axes = plt.subplots(nrows=number_rows, ncols=plots_each_row, figsize=(16, 8),) # sharey=True
        axes = np.array(axes).reshape(number_rows, plots_each_row)
        coordinates = list(itertools.product(range(number_rows), range(plots_each_row)))

        for coor, (estimator_name, estimator) in zip(coordinates, estimators_dict.items()):
            y_score = estimator.predict_proba(self.x)[:, 1]
            fpr, tpr, thresholds = roc_curve(self.y, y_score)
            close_zero = np.argmin(np.abs(thresholds - 0.5))
            
            axes[coor].plot(fpr, tpr, label=f"{estimator_name}'s ROC Curve")
            axes[coor].set_xlabel("FPR")
            axes[coor].set_ylabel("TPR (recall)")
            axes[coor].plot(fpr[close_zero + 1], tpr[close_zero + 1], 'o', markersize=4, label=f"{estimator_name} default threshold", fillstyle="none", c='k', mew=2)
            axes[coor].plot(np.linspace(0, 1, 100), np.linspace(0, 1, 100), "b--", label=f"Random Classifier")
            axes[coor].legend(loc='best')
            axes[coor].grid()

        # hide any unused axes
        for coor in coordinates[num_plots:]:
            fig.delaxes(axes[coor])

    def compute_auc_score(self, estimators_dict:dict, tree_models:bool):
        estimators_score = {}
        for estimator_name, estimator in estimators_dict.items():
            y_score = estimator.predict_proba(self.x)[:, 1] # certainty measure for the positive class (class 1)
            auc_score = roc_auc_score(self.y, y_score)
            estimators_score[estimator_name] = auc_score
        return estimators_score

In [41]:
roc_curve_plotter = ROCCurve(X_test_copy, y_test)
xg_tuned_model = xg_best_model['best_estimator']

estimators_dict={
    'xg_boost': xg_tuned_model
}
tree_models=True

score = roc_curve_plotter.compute_auc_score(estimators_dict, tree_models)
print(f"Điểm AUC của Decision Tree là: {score['xg_boost']}")

# Điểm AUC cao -> mô hình rất tốt trong việc dự đoán positive đối với những người thật sự là positive
# cũng như ít nhầm lẫn trong việc dự đoán positive đối với những người thật sự là negative

Điểm AUC của Decision Tree là: 0.7256680687114228


In [ ]:
# fine-tune cho random forest
param_distributions = {'rf__max_depth': [None],
                       'rf__min_samples_leaf': [5, 10, 20, 50],
                       'rf__min_samples_split': [10, 20, 50, 100],
                       'rf__max_features': ['log2', 'sqrt', 0.3],
                       'rf__n_estimators': [75, 150, 200],
                       'rf__class_weight': [{0: 1, 1: 6}, {0: 1, 1: 8}, {0: 1, 1: 10}]
                      }

scoring = {'f1': 'f1', 'recall': 'recall', 'precision': 'precision', 'neg_log_loss': 'neg_log_loss'}

rf_best_model, rf_table = searcher.random_search_cv(estimator=rf_pipeline, param_distributions=param_distributions, 
                                               scoring=scoring, cv='StratifiedKFold', refit='neg_log_loss')
rf_best_model

In [ ]:
rf_table

In [44]:
# fine tune cho ada boost
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)
class_weights = compute_class_weight(
    class_weight={0: 1, 1: 8},
    classes=classes,
    y=y_train
)
class_weight_dict = dict(zip(classes, class_weights))
sample_weight = np.array([class_weight_dict[label] for label in y_train])


param_distributions = {
    "ada__n_estimators": [75, 150, 225, 300],
    "ada__learning_rate": [1.0, 0.5, 0.1, 0.01],
}

scoring = {'f1': 'f1', 'recall': 'recall', 'precision': 'precision', 'neg_log_loss': 'neg_log_loss'}

fit_params = {"ada__sample_weight": sample_weight}
ada_best_model, ada_table = searcher.random_search_cv(estimator=ada_pipeline, param_distributions=param_distributions, 
                                               scoring=scoring, cv='StratifiedKFold', refit='neg_log_loss', fit_params=fit_params)

ada_best_model

ValueError: 
All the 30 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
30 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py", line 662, in fit
    self._final_estimator.fit(Xt, y, **last_step_params["fit"])
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
   

In [ ]:
ada_table

In [72]:
final_pipeline = FinalPipeline()
final_dataset_train_copy = final_dataset_train.copy()
final_dataset_train_copy = final_pipeline.fit_transform(final_dataset_train_copy)
filter_ = [col for col in final_dataset_train_copy.columns if col.startswith('AT_NAME_EDUCATION_TYPE')]
final_dataset_train_copy[filter_]

,AT_NAME_EDUCATION_TYPE_Higher education,AT_NAME_EDUCATION_TYPE_Incomplete higher,AT_NAME_EDUCATION_TYPE_Secondary / secondary special
0,0.0,0.0,1.0
1,0.0,0.0,1.0
2,0.0,0.0,1.0
3,0.0,0.0,1.0
4,0.0,0.0,1.0
...,...,...,...
39356,1.0,0.0,0.0
39357,1.0,0.0,0.0
39358,1.0,0.0,0.0
39359,0.0,0.0,1.0


In [63]:
number_columns = final_dataset_train_copy.select_dtypes('number')
nunique = number_columns.nunique() 
binary_columns = list(nunique[nunique==2].index)
binary_means = final_dataset_train_copy[binary_columns].mean()
binary_means.sort_values(ascending=True)[:12]

CC_NAME_CONTRACT_STATUS_Demand_MEAN_SUM                   0.000002
PC_NAME_CONTRACT_STATUS_Amortized debt_MEAN_SUM           0.000008
PC_NAME_CONTRACT_STATUS_Amortized debt_MEAN_MEAN          0.000009
PC_NAME_CONTRACT_STATUS_Amortized debt_MEAN_MIN           0.000009
PC_NAME_CONTRACT_STATUS_Amortized debt_MEAN_MAX           0.000009
AT_NAME_INCOME_TYPE_Businessman                           0.000025
AT_NAME_FAMILY_STATUS_Unknown                             0.000025
AT_NAME_INCOME_TYPE_Student                               0.000025
CC_NAME_CONTRACT_STATUS_Demand_SUM_SUM                    0.000027
PA_NAME_CASH_LOAN_PURPOSE_Money for a third person_SUM    0.000080
PC_NAME_CONTRACT_STATUS_Canceled_SUM_SUM                  0.000080
PC_NAME_CONTRACT_STATUS_Canceled_SUM_MAX                  0.000082
AT_NAME_INCOME_TYPE_Unemployed                            0.000127
dtype: float64